# Quickstart to demonstrate an OCP agent

In [3]:
!pip install deepagents langchain_openai langchain_mcp_adapters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 669.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 687.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 214.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 566.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 220.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Uninstalling pydantic_core-2.27.2:
      Successfully uninstalled pydantic_core-2.27.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.10.6
    Uninstalling pydantic-2.10.6:
      Successfully uninstalled pydantic-2.10.6
  Attempting 

## Create MCP tool Client

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient  
import os

In [11]:
ocp_mcp_endpoint = os.environ["OCP_MCP_ENDPOINT"]

mcp_servers = {
        "openshift-mcp": {
            "transport": "http",
            "url": ocp_mcp_endpoint,
        }
    }

print(mcp_servers)

mcp_client = MultiServerMCPClient(mcp_servers)

            

{'openshift-mcp': {'transport': 'http', 'url': 'https://ocp-mcp.apps.cluster-hfsk9.hfsk9.sandbox160.opentlc.com/mcp'}}


# Create the Agent

In [12]:
from langchain_openai import ChatOpenAI

qwen = ChatOpenAI(
    base_url=os.environ["QWEN_ENDPOINT"],
    api_key=os.environ["QWEN_API_KEY"],
    model="Qwen3.6-35B-A3B",
)

In [13]:
research_instruction = """You are an expert researcher. Your job is to gather information about a connected OpenShift cluster. Use the MCP tools at your disposal to accomplish this
"""

In [14]:
from deepagents import create_deep_agent
import asyncio

tools = await mcp_client.get_tools()
agent = create_deep_agent(
    model=qwen,
    tools=tools,
    system_prompt=research_instruction,
)

# Test the agent

In [15]:
result = await agent.ainvoke({"messages": [{"role": "user", "content": "List the pods currently running on the cluster"}]})

# Print the agent's response
print(result["messages"][-1].content)



The cluster has a large number of pods. Here's a summary grouped by namespace:

## Cluster Pod Summary

| Namespace | Running Pods | Status Notes |
|-----------|-------------|--------------|
| **cert-manager** | 3 | controller, cainjector, webhook |
| **cert-manager-operator** | 1 | controller-manager |
| **deepagent** | 1 | quickstart notebook pod |
| **istio-system** | 3 | egress, ingress, istiod |
| **knative-serving** | 12 | activator, autoscaler, controller, net-istio, webhook |
| **kube-system** | 1 | sealed-secrets |
| **nvidia-gpu-operator** | 8 | gpu-discovery, device-plugin, DCGM, driver, validator (1 completed) |
| **ocp-mcp-server** | 1 | kubernetes-mcp-server |
| **openshift-apiserver** | 3 | api servers |
| **openshift-apiserver-operator** | 1 | |
| **openshift-authentication** | 3 | oauth |
| **openshift-authentication-operator** | 1 | |
| **openshift-catalogd** | 1 | catalogd manager |
| **openshift-cloud-controller-manager** | 2 | AWS |
| **openshift-cloud-controller